# 03 Verified Baseline

This notebook creates the verified DFSI baseline using the corrected additive calculation from `02d_dfsi_expression_audit_CORRECTED_V2.ipynb`. It reloads raw CSV files directly and does not use calculations or outputs from older incorrect notebooks. It does not modify `data/raw` and does not start Monte Carlo sensitivity analysis.

## Setup

In [1]:
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 180)

EXPECTED_RAW_FILES = [
    "DFSI.csv",
    "District_FloodedArea.csv",
    "District_FloodImpact.csv",
    "India_Flood_Inventory_v3.csv",
]


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        raw_dir = candidate / "data" / "raw"
        if all((raw_dir / file_name).exists() for file_name in EXPECTED_RAW_FILES):
            return candidate
    raise FileNotFoundError("Could not find repository root containing data/raw with expected CSV files.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/abhijithshaji/Documents/india-flood-index-robustness


## Reload Raw Sources

In [2]:
dfsi_raw = pd.read_csv(RAW_DIR / "DFSI.csv", low_memory=False)
impact_raw = pd.read_csv(RAW_DIR / "District_FloodImpact.csv", low_memory=False)
area_raw = pd.read_csv(RAW_DIR / "District_FloodedArea.csv", low_memory=False)
inventory_raw = pd.read_csv(RAW_DIR / "India_Flood_Inventory_v3.csv", low_memory=False)

for name, table in {
    "DFSI.csv": dfsi_raw,
    "District_FloodImpact.csv": impact_raw,
    "District_FloodedArea.csv": area_raw,
    "India_Flood_Inventory_v3.csv": inventory_raw,
}.items():
    print(f"{name}: {table.shape[0]:,} rows x {table.shape[1]:,} columns")
    print(list(table.columns))

DFSI.csv: 744 rows x 3 columns
['Unnamed: 0', 'State_Name', 'DFSI']
District_FloodImpact.csv: 732 rows x 5 columns
['Dist_Name', 'Human_fatality', 'Human_injured', 'Population', 'Mean_Flood_Duration']
District_FloodedArea.csv: 732 rows x 4 columns
['Dist_Name', 'Percent_Flooded_Area', 'Parmanent_Water', 'Corrected_Percent_Flooded_Area']
India_Flood_Inventory_v3.csv: 6,876 rows x 23 columns
['Unnamed: 0', 'UEI', 'Start Date', 'End Date', 'Duration(Days)', 'Main Cause', 'Location', 'Districts', 'State', 'Latitude', 'Longitude', 'Severity', 'Area Affected', 'Human fatality', 'Human injured', 'Human Displaced', 'Animal Fatality', 'Description of Casualties/injured', 'Extent of damage ', 'Event Source', 'Event Souce ID', 'District_LGD_Codes', 'State_Codes']


## Component Mapping

In [3]:
def make_review_key(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", str(value).strip()).casefold()


def split_list_cell(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",")]


def clean_lgd_code(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.casefold() in {"none", "nan", "na", "<na>"}:
        return pd.NA
    return text


component_mapping = pd.DataFrame([
    {"baseline_column": "flood_event_count", "symbol": "E", "raw_source": "India_Flood_Inventory_v3.csv", "raw_column": "distinct UEI after exploding Districts", "notes": "LGD district codes are retained where available; no matched events are counted as zero events."},
    {"baseline_column": "mean_flood_duration", "symbol": "D", "raw_source": "District_FloodImpact.csv", "raw_column": "Mean_Flood_Duration", "notes": "Converted to float64; missing values remain missing."},
    {"baseline_column": "fatalities", "symbol": "F", "raw_source": "District_FloodImpact.csv", "raw_column": "Human_fatality", "notes": "Converted to float64."},
    {"baseline_column": "injured", "symbol": "I", "raw_source": "District_FloodImpact.csv", "raw_column": "Human_injured", "notes": "Converted to float64."},
    {"baseline_column": "corrected_percent_flooded_area", "symbol": "A", "raw_source": "District_FloodedArea.csv", "raw_column": "Corrected_Percent_Flooded_Area", "notes": "Paper-supported corrected percent flooded area."},
    {"baseline_column": "population", "symbol": "P", "raw_source": "District_FloodImpact.csv", "raw_column": "Population", "notes": "Converted to float64; enters as log10(P)."},
])
display(component_mapping)

,baseline_column,symbol,raw_source,raw_column,notes
0,flood_event_count,E,India_Flood_Inventory_v3.csv,distinct UEI after exploding Districts,LGD district codes are retained where available; no matched events are counted as zero events.
1,mean_flood_duration,D,District_FloodImpact.csv,Mean_Flood_Duration,Converted to float64; missing values remain missing.
2,fatalities,F,District_FloodImpact.csv,Human_fatality,Converted to float64.
3,injured,I,District_FloodImpact.csv,Human_injured,Converted to float64.
4,corrected_percent_flooded_area,A,District_FloodedArea.csv,Corrected_Percent_Flooded_Area,Paper-supported corrected percent flooded area.
5,population,P,District_FloodImpact.csv,Population,Converted to float64; enters as log10(P).


## Rebuild Joins and Identifier Audit

Uniqueness is checked with all identifiers available in each source. The impact and flooded-area files lack state/LGD identifiers, so repeated district names are not silently discarded; they are preserved in `dfsi_unresolved_records.csv`.

In [4]:
event_rows = []
for _, row in inventory_raw.reset_index(names="inventory_source_row_id").iterrows():
    districts = split_list_cell(row["Districts"])
    district_codes = split_list_cell(row["District_LGD_Codes"])
    for position, district_name in enumerate(districts):
        event_rows.append({
            "inventory_source_row_id": row["inventory_source_row_id"],
            "UEI": row.get("UEI"),
            "district_name_from_inventory": district_name,
            "district_review_key": make_review_key(district_name),
            "district_lgd_code": clean_lgd_code(district_codes[position] if position < len(district_codes) else pd.NA),
            "event_state_text": row.get("State"),
            "event_state_codes_text": row.get("State_Codes"),
        })

event_districts = pd.DataFrame(event_rows)
event_counts = (
    event_districts.groupby("district_review_key", dropna=False)
    .agg(
        flood_event_count=("UEI", "nunique"),
        inventory_district_names=("district_name_from_inventory", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        available_district_lgd_codes=("district_lgd_code", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        available_district_lgd_code_count=("district_lgd_code", lambda s: int(s.dropna().nunique())),
    )
    .reset_index()
)

impact = impact_raw.reset_index(names="impact_source_row_id").copy()
impact["district_review_key"] = impact["Dist_Name"].map(make_review_key)
area = area_raw.reset_index(names="area_source_row_id").copy()
area["district_review_key"] = area["Dist_Name"].map(make_review_key)
published = dfsi_raw.reset_index(names="dfsi_source_row_id").rename(columns={"Unnamed: 0": "district_name"}).copy()
published["district_review_key"] = published["district_name"].map(make_review_key)
published["state_review_key"] = published["State_Name"].map(make_review_key)
published["published_dfsi"] = pd.to_numeric(published["DFSI"], errors="coerce").astype("float64")

impact_counts = impact["district_review_key"].value_counts(dropna=False).rename("impact_count")
area_counts = area["district_review_key"].value_counts(dropna=False).rename("area_count")
published_counts = published["district_review_key"].value_counts(dropna=False).rename("published_count")
event_counts_by_key = event_counts.set_index("district_review_key")["flood_event_count"].rename("event_key_count")

key_audit = (
    pd.concat([impact_counts, area_counts, published_counts, event_counts_by_key], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"index": "district_review_key"})
)

key_audit["is_one_to_one_comparison_key"] = key_audit["impact_count"].eq(1) & key_audit["area_count"].eq(1) & key_audit["published_count"].eq(1)
key_audit["has_repeated_name"] = key_audit[["impact_count", "area_count", "published_count"]].gt(1).any(axis=1)
key_audit["has_unmatched_required_source"] = key_audit[["impact_count", "area_count", "published_count"]].eq(0).any(axis=1)

one_to_one_keys = key_audit.loc[key_audit["is_one_to_one_comparison_key"], "district_review_key"]

baseline_work = (
    impact[impact["district_review_key"].isin(one_to_one_keys)]
    .merge(area[area["district_review_key"].isin(one_to_one_keys)], on="district_review_key", how="inner", suffixes=("_impact", "_area"), validate="one_to_one")
    .merge(event_counts, on="district_review_key", how="left", validate="one_to_one")
    .merge(published[published["district_review_key"].isin(one_to_one_keys)][[
        "dfsi_source_row_id", "district_name", "State_Name", "state_review_key", "published_dfsi", "district_review_key"
    ]], on="district_review_key", how="inner", validate="one_to_one")
)
baseline_work["flood_event_count_missing_from_inventory"] = baseline_work["flood_event_count"].isna()
baseline_work["flood_event_count"] = baseline_work["flood_event_count"].fillna(0)

print(f"Current one-to-one comparison records: {len(baseline_work):,}")
print(f"Repeated district-name keys retained in unresolved audit: {int(key_audit['has_repeated_name'].sum()):,}")
print(f"Required-source unmatched keys retained in unresolved audit: {int(key_audit['has_unmatched_required_source'].sum()):,}")

Current one-to-one comparison records: 720
Repeated district-name keys retained in unresolved audit: 6
Required-source unmatched keys retained in unresolved audit: 237


## Corrected Additive DFSI

The reconstructed value is the sum of all six log terms, and it is independently checked against the equivalent product-space calculation.

In [5]:
baseline = baseline_work.copy()

baseline["flood_event_count"] = pd.to_numeric(baseline["flood_event_count"], errors="coerce").astype("float64")
baseline["mean_flood_duration"] = pd.to_numeric(baseline["Mean_Flood_Duration"], errors="coerce").astype("float64")
baseline["fatalities"] = pd.to_numeric(baseline["Human_fatality"], errors="coerce").astype("float64")
baseline["injured"] = pd.to_numeric(baseline["Human_injured"], errors="coerce").astype("float64")
baseline["corrected_percent_flooded_area"] = pd.to_numeric(baseline["Corrected_Percent_Flooded_Area"], errors="coerce").astype("float64")
baseline["population"] = pd.to_numeric(baseline["Population"], errors="coerce").astype("float64")

component_columns = [
    "flood_event_count",
    "mean_flood_duration",
    "fatalities",
    "injured",
    "corrected_percent_flooded_area",
    "population",
]
complete_mask = baseline[component_columns].notna().all(axis=1)

for column in component_columns:
    assert baseline.loc[complete_mask, column].ge(0).all(), f"Negative component in {column}"
assert baseline.loc[complete_mask, "population"].gt(0).all(), "Population must be greater than zero"

baseline["term_events"] = np.log10(1.0 + baseline["flood_event_count"])
baseline["term_duration"] = np.log10(1.0 + baseline["mean_flood_duration"])
baseline["term_fatalities"] = 2.0 * np.log10(1.0 + baseline["fatalities"])
baseline["term_injured"] = np.log10(1.0 + baseline["injured"])
baseline["term_flooded_area"] = np.log10(1.0 + baseline["corrected_percent_flooded_area"])
baseline["term_population"] = np.log10(baseline["population"])

baseline["reconstructed_dfsi"] = (
    baseline["term_events"]
    + baseline["term_duration"]
    + baseline["term_fatalities"]
    + baseline["term_injured"]
    + baseline["term_flooded_area"]
    + baseline["term_population"]
)

baseline["product_space_value"] = (
    (1.0 + baseline["flood_event_count"])
    * (1.0 + baseline["mean_flood_duration"])
    * (1.0 + baseline["fatalities"]) ** 2.0
    * (1.0 + baseline["injured"])
    * (1.0 + baseline["corrected_percent_flooded_area"])
    * baseline["population"]
)
baseline["reconstructed_dfsi_product_space"] = np.log10(baseline["product_space_value"])
baseline["log_product_agreement_abs_diff"] = (baseline["reconstructed_dfsi"] - baseline["reconstructed_dfsi_product_space"]).abs()

assert baseline.loc[complete_mask, "reconstructed_dfsi"].gt(0).all(), "All reconstructed DFSI values must be positive for complete rows"
assert baseline.loc[complete_mask, "log_product_agreement_abs_diff"].le(1e-10).all(), "Log-space and product-space calculations disagree"

baseline["completeness_status"] = np.where(complete_mask, "complete", "incomplete_missing_component")
baseline["join_status"] = "one_to_one_comparison_record"

baseline["published_rank"] = baseline["published_dfsi"].rank(ascending=False, method="min")
baseline.loc[baseline["published_dfsi"].isna(), "published_rank"] = np.nan
baseline["reconstructed_rank"] = baseline["reconstructed_dfsi"].rank(ascending=False, method="min")
baseline.loc[baseline["reconstructed_dfsi"].isna(), "reconstructed_rank"] = np.nan

assert baseline.loc[baseline["reconstructed_rank"].eq(1), "reconstructed_dfsi"].eq(baseline["reconstructed_dfsi"].max()).all(), "Reconstructed rank 1 is not maximum reconstructed DFSI"
assert baseline.loc[baseline["published_rank"].eq(1), "published_dfsi"].eq(baseline["published_dfsi"].max()).all(), "Published rank 1 is not maximum published DFSI"

baseline["numerical_difference"] = baseline["reconstructed_dfsi"] - baseline["published_dfsi"]
baseline["absolute_difference"] = baseline["numerical_difference"].abs()
baseline["rank_difference"] = baseline["reconstructed_rank"] - baseline["published_rank"]
baseline["absolute_rank_difference"] = baseline["rank_difference"].abs()

baseline["available_state_identifier"] = baseline["State_Name"]
baseline["available_state_review_key"] = baseline["state_review_key"]
baseline["available_district_identifier"] = baseline["district_name"]
baseline["available_district_review_key"] = baseline["district_review_key"]
baseline["available_district_lgd_codes"] = baseline["available_district_lgd_codes"].fillna("")
baseline["available_district_lgd_code_count"] = baseline["available_district_lgd_code_count"].fillna(0).astype(int)

print(f"Complete rows: {int(complete_mask.sum()):,}")
print(f"Incomplete rows: {int((~complete_mask).sum()):,}")
print(f"Max log/product absolute difference: {baseline.loc[complete_mask, 'log_product_agreement_abs_diff'].max()}")

Complete rows: 709
Incomplete rows: 11
Max log/product absolute difference: 3.552713678800501e-15


## Save Verified Baseline

In [6]:
baseline_columns = [
    "State_Name",
    "district_name",
    "available_state_identifier",
    "available_state_review_key",
    "available_district_identifier",
    "available_district_review_key",
    "available_district_lgd_codes",
    "available_district_lgd_code_count",
    "inventory_district_names",
    "dfsi_source_row_id",
    "impact_source_row_id",
    "area_source_row_id",
    "flood_event_count",
    "mean_flood_duration",
    "fatalities",
    "injured",
    "corrected_percent_flooded_area",
    "population",
    "published_dfsi",
    "reconstructed_dfsi",
    "reconstructed_dfsi_product_space",
    "log_product_agreement_abs_diff",
    "published_rank",
    "reconstructed_rank",
    "numerical_difference",
    "absolute_difference",
    "rank_difference",
    "absolute_rank_difference",
    "completeness_status",
    "join_status",
]
verified_baseline = baseline[baseline_columns].copy()
verified_baseline.to_csv(PROCESSED_DIR / "dfsi_verified_baseline.csv", index=False)
display(verified_baseline.head())
print(f"Saved {PROCESSED_DIR / 'dfsi_verified_baseline.csv'}")

,State_Name,district_name,available_state_identifier,available_state_review_key,available_district_identifier,available_district_review_key,available_district_lgd_codes,available_district_lgd_code_count,inventory_district_names,dfsi_source_row_id,impact_source_row_id,area_source_row_id,flood_event_count,mean_flood_duration,fatalities,injured,corrected_percent_flooded_area,population,published_dfsi,reconstructed_dfsi,reconstructed_dfsi_product_space,log_product_agreement_abs_diff,published_rank,reconstructed_rank,numerical_difference,absolute_difference,rank_difference,absolute_rank_difference,completeness_status,join_status
0,DADRA & NAGAR HAVE,Dadra & Nagar Haveli,DADRA & NAGAR HAVE,dadra & nagar have,Dadra & Nagar Haveli,dadra & nagar haveli,,0,NaN,653,0,0,0.0,1.0,10.0,0.0,2.969168,618459.0,9.820211,8.773826,8.773826,0.0,630.0,637.0,-1.046385,1.046385,7.0,7.0,complete,one_to_one_comparison_record
1,DAMAN & DIU,Daman,DAMAN & DIU,daman & diu,Daman,daman,,0,NaN,738,1,1,0.0,NaN,0.0,0.0,1.544538,190125.0,5.233751,NaN,NaN,NaN,715.0,NaN,NaN,NaN,NaN,NaN,incomplete_missing_component,one_to_one_comparison_record
2,DAMAN & DIU,Diu,DAMAN & DIU,daman & diu,Diu,diu,,0,NaN,741,2,2,0.0,NaN,0.0,0.0,0.819836,67412.0,4.269142,NaN,NaN,NaN,718.0,NaN,NaN,NaN,NaN,NaN,incomplete_missing_component,one_to_one_comparison_record
3,ANDAMAN & NICOBAR,North & Middle Andaman,ANDAMAN & NICOBAR,andaman & nicobar,North & Middle Andaman,north & middle andaman,,0,NaN,726,3,3,0.0,1.0,0.0,0.0,1.314817,101677.0,7.462575,5.672769,5.672769,0.0,703.0,709.0,-1.789805,1.789805,6.0,6.0,complete,one_to_one_comparison_record
4,ANDAMAN & NICOBAR,South Andaman,ANDAMAN & NICOBAR,andaman & nicobar,South Andaman,south andaman,,0,NaN,703,4,4,0.0,2.0,1.0,0.0,1.443452,257900.0,8.360463,6.878636,6.878636,0.0,680.0,697.0,-1.481827,1.481827,17.0,17.0,complete,one_to_one_comparison_record


Saved /Users/abhijithshaji/Documents/india-flood-index-robustness/data/processed/dfsi_verified_baseline.csv


## Preserve Unresolved and Incomplete Records

In [7]:
unresolved_records = []

for row in key_audit.itertuples(index=False):
    notes = []
    if row.impact_count > 1:
        notes.append("repeated in District_FloodImpact.csv")
    if row.area_count > 1:
        notes.append("repeated in District_FloodedArea.csv")
    if row.published_count > 1:
        notes.append("repeated in DFSI.csv by district name despite state identifiers")
    if row.impact_count == 0:
        notes.append("missing from District_FloodImpact.csv")
    if row.area_count == 0:
        notes.append("missing from District_FloodedArea.csv")
    if row.published_count == 0:
        notes.append("missing from DFSI.csv")
    if notes:
        unresolved_records.append({
            "district_review_key": row.district_review_key,
            "State_Name": pd.NA,
            "district_name": pd.NA,
            "issue_type": "join_key_unresolved",
            "impact_count": row.impact_count,
            "area_count": row.area_count,
            "published_count": row.published_count,
            "event_key_count": row.event_key_count,
            "completeness_status": pd.NA,
            "join_status": "not_one_to_one_or_unmatched",
            "details": "; ".join(notes),
        })

for row in verified_baseline.query("completeness_status != 'complete'").itertuples(index=False):
    missing_components = [column for column in component_columns if pd.isna(getattr(row, column))]
    unresolved_records.append({
        "district_review_key": row.available_district_review_key,
        "State_Name": row.State_Name,
        "district_name": row.district_name,
        "issue_type": "incomplete_comparison_record",
        "impact_count": 1,
        "area_count": 1,
        "published_count": 1,
        "event_key_count": row.flood_event_count,
        "completeness_status": row.completeness_status,
        "join_status": row.join_status,
        "details": "Missing component(s): " + ", ".join(missing_components),
    })

unresolved_df = pd.DataFrame(unresolved_records).sort_values(["issue_type", "district_review_key"], na_position="last")
unresolved_df.to_csv(TABLE_DIR / "dfsi_unresolved_records.csv", index=False)
display(unresolved_df.head(30))
print(f"Unresolved records: {len(unresolved_df):,}")

,district_review_key,State_Name,district_name,issue_type,impact_count,area_count,published_count,event_key_count,completeness_status,join_status,details
252,almora,UTTARAKHAND,Almora\n,incomplete_comparison_record,1,1,1,20.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
246,dakshin bastar dantewada,CHHATTISGARH,Dakshin Bastar Dantewada,incomplete_comparison_record,1,1,1,2.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
243,daman,DAMAN & DIU,Daman,incomplete_comparison_record,1,1,1,0.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
244,diu,DAMAN & DIU,Diu,incomplete_comparison_record,1,1,1,0.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
250,east,DELHI,East,incomplete_comparison_record,1,1,1,19.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
248,gariaband,CHHATTISGARH,Gariaband,incomplete_comparison_record,1,1,1,0.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
253,jhargram,WEST BENGAL,Jhargram,incomplete_comparison_record,1,1,1,0.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
247,kondagaon,CHHATTISGARH,Kondagaon,incomplete_comparison_record,1,1,1,0.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
251,lakshadweep,LAKSHADWEEP,Lakshadweep,incomplete_comparison_record,1,1,1,0.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration
245,nicobars,ANDAMAN & NICOBAR,Nicobars,incomplete_comparison_record,1,1,1,0.0,incomplete_missing_component,one_to_one_comparison_record,Missing component(s): mean_flood_duration


Unresolved records: 254


## Numerical vs Ranking Reproduction Summary

In [8]:
complete_baseline = verified_baseline.query("completeness_status == 'complete'").copy()
complete_baseline = complete_baseline.replace([np.inf, -np.inf], np.nan).dropna(subset=["published_dfsi", "reconstructed_dfsi", "published_rank", "reconstructed_rank"])

spearman_corr = complete_baseline["reconstructed_dfsi"].corr(complete_baseline["published_dfsi"], method="spearman")
pearson_corr = complete_baseline["reconstructed_dfsi"].corr(complete_baseline["published_dfsi"], method="pearson")
median_abs_num_diff = complete_baseline["absolute_difference"].median()
median_abs_rank_diff = complete_baseline["absolute_rank_difference"].median()
max_abs_rank_diff = complete_baseline["absolute_rank_difference"].max()

summary_records = [
    {"section": "numerical_reproduction", "metric": "one_to_one_comparison_rows", "value": len(verified_baseline), "notes": "All current one-to-one comparison records, including incomplete rows."},
    {"section": "numerical_reproduction", "metric": "complete_comparison_rows", "value": len(complete_baseline), "notes": "Rows with all components and both DFSI values available."},
    {"section": "numerical_reproduction", "metric": "incomplete_records_preserved", "value": int((verified_baseline["completeness_status"] != "complete").sum()), "notes": "Missing components are not imputed."},
    {"section": "numerical_reproduction", "metric": "spearman_correlation", "value": spearman_corr, "notes": "Correlation between reconstructed and published DFSI scores."},
    {"section": "numerical_reproduction", "metric": "pearson_correlation", "value": pearson_corr, "notes": "Correlation between reconstructed and published DFSI scores."},
    {"section": "numerical_reproduction", "metric": "median_absolute_numerical_difference", "value": median_abs_num_diff, "notes": "Median absolute difference between reconstructed and published DFSI scores."},
    {"section": "ranking_reproduction", "metric": "median_absolute_rank_difference", "value": median_abs_rank_diff, "notes": "Median absolute difference between descending published and reconstructed ranks."},
    {"section": "ranking_reproduction", "metric": "maximum_absolute_rank_difference", "value": max_abs_rank_diff, "notes": "Maximum absolute rank change among complete comparison rows."},
    {"section": "assertions", "metric": "all_reconstructed_dfsi_values_positive", "value": bool(complete_baseline["reconstructed_dfsi"].gt(0).all()), "notes": "Assertion passed during calculation."},
    {"section": "assertions", "metric": "corrected_log_space_product_space_agree", "value": bool(complete_baseline["log_product_agreement_abs_diff"].le(1e-10).all()), "notes": "Assertion passed during calculation."},
    {"section": "assertions", "metric": "reconstructed_rank_1_has_maximum_reconstructed_dfsi", "value": bool(complete_baseline.loc[complete_baseline["reconstructed_rank"].eq(1), "reconstructed_dfsi"].eq(complete_baseline["reconstructed_dfsi"].max()).all()), "notes": "Descending rank check."},
    {"section": "assertions", "metric": "published_rank_1_has_maximum_published_dfsi", "value": bool(complete_baseline.loc[complete_baseline["published_rank"].eq(1), "published_dfsi"].eq(complete_baseline["published_dfsi"].max()).all()), "notes": "Descending rank check."},
]

discrepancy_summary = pd.DataFrame(summary_records)
discrepancy_summary.to_csv(TABLE_DIR / "dfsi_verified_discrepancy_summary.csv", index=False)
display(discrepancy_summary)

,section,metric,value,notes
0,numerical_reproduction,one_to_one_comparison_rows,720,"All current one-to-one comparison records, including incomplete rows."
1,numerical_reproduction,complete_comparison_rows,709,Rows with all components and both DFSI values available.
2,numerical_reproduction,incomplete_records_preserved,11,Missing components are not imputed.
3,numerical_reproduction,spearman_correlation,0.958847,Correlation between reconstructed and published DFSI scores.
4,numerical_reproduction,pearson_correlation,0.958716,Correlation between reconstructed and published DFSI scores.
5,numerical_reproduction,median_absolute_numerical_difference,1.206769,Median absolute difference between reconstructed and published DFSI scores.
6,ranking_reproduction,median_absolute_rank_difference,18.0,Median absolute difference between descending published and reconstructed ranks.
7,ranking_reproduction,maximum_absolute_rank_difference,345.0,Maximum absolute rank change among complete comparison rows.
8,assertions,all_reconstructed_dfsi_values_positive,True,Assertion passed during calculation.
9,assertions,corrected_log_space_product_space_agree,True,Assertion passed during calculation.


## Top-Group Overlap

In [9]:
def top_group_overlap(frame, group_name, n):
    published_top = set(frame.nsmallest(n, "published_rank")["available_district_review_key"])
    reconstructed_top = set(frame.nsmallest(n, "reconstructed_rank")["available_district_review_key"])
    overlap = published_top & reconstructed_top
    return {
        "group": group_name,
        "n_requested": n,
        "published_group_size": len(published_top),
        "reconstructed_group_size": len(reconstructed_top),
        "overlap_count": len(overlap),
        "published_overlap_share": len(overlap) / len(published_top) if published_top else np.nan,
        "reconstructed_overlap_share": len(overlap) / len(reconstructed_top) if reconstructed_top else np.nan,
        "jaccard_overlap": len(overlap) / len(published_top | reconstructed_top) if (published_top | reconstructed_top) else np.nan,
        "published_only_count": len(published_top - reconstructed_top),
        "reconstructed_only_count": len(reconstructed_top - published_top),
    }

n_complete = len(complete_baseline)
top_10_percent_n = math.ceil(n_complete * 0.10)
overlap_summary = pd.DataFrame([
    top_group_overlap(complete_baseline, "top_10_percent", top_10_percent_n),
    top_group_overlap(complete_baseline, "top_25_districts", min(25, n_complete)),
    top_group_overlap(complete_baseline, "top_50_districts", min(50, n_complete)),
    top_group_overlap(complete_baseline, "top_100_districts", min(100, n_complete)),
])
overlap_summary.to_csv(TABLE_DIR / "dfsi_verified_top_group_overlap.csv", index=False)
display(overlap_summary)

,group,n_requested,published_group_size,reconstructed_group_size,overlap_count,published_overlap_share,reconstructed_overlap_share,jaccard_overlap,published_only_count,reconstructed_only_count
0,top_10_percent,71,71,71,59,0.830986,0.830986,0.710843,12,12
1,top_25_districts,25,25,25,19,0.760000,0.760000,0.612903,6,6
2,top_50_districts,50,50,50,41,0.820000,0.820000,0.694915,9,9
3,top_100_districts,100,100,100,89,0.890000,0.890000,0.801802,11,11


## Largest Rank Changes

In [10]:
rank_changes = complete_baseline.loc[complete_baseline["absolute_rank_difference"].gt(10)].copy()
rank_changes["rank_change_gt_10"] = rank_changes["absolute_rank_difference"].gt(10)
rank_changes["rank_change_gt_25"] = rank_changes["absolute_rank_difference"].gt(25)
rank_changes["rank_change_gt_50"] = rank_changes["absolute_rank_difference"].gt(50)
rank_changes = rank_changes.sort_values("absolute_rank_difference", ascending=False)
rank_change_columns = [
    "State_Name",
    "district_name",
    "available_district_review_key",
    "published_dfsi",
    "reconstructed_dfsi",
    "published_rank",
    "reconstructed_rank",
    "rank_difference",
    "absolute_rank_difference",
    "numerical_difference",
    "absolute_difference",
    "rank_change_gt_10",
    "rank_change_gt_25",
    "rank_change_gt_50",
]
rank_changes[rank_change_columns].to_csv(TABLE_DIR / "dfsi_verified_largest_rank_changes.csv", index=False)
display(rank_changes[rank_change_columns].head(50))
print("Rank changes > 10:", int(rank_changes["rank_change_gt_10"].sum()))
print("Rank changes > 25:", int(rank_changes["rank_change_gt_25"].sum()))
print("Rank changes > 50:", int(rank_changes["rank_change_gt_50"].sum()))

,State_Name,district_name,available_district_review_key,published_dfsi,reconstructed_dfsi,published_rank,reconstructed_rank,rank_difference,absolute_rank_difference,numerical_difference,absolute_difference,rank_change_gt_10,rank_change_gt_25,rank_change_gt_50
395,MAHARASHTRA,Mumbai Suburban,mumbai suburban,12.892295,15.303536,452.0,107.0,-345.0,345.0,2.411241,2.411241,True,True,True
535,TELANGANA,Hyderabad,hyderabad,10.461370,13.745869,591.0,248.0,-343.0,343.0,3.284499,3.284499,True,True,True
334,MADHYA PRADESH,Bhopal,bhopal,11.677641,13.592510,521.0,267.0,-254.0,254.0,1.914869,1.914869,True,True,True
370,MAHARASHTRA,Washim,washim,10.734975,13.174830,580.0,330.0,-250.0,250.0,2.439856,2.439856,True,True,True
273,KARNATAKA,Uttara Kannada,uttara kannada,16.679683,12.967175,106.0,350.0,244.0,244.0,-3.712508,3.712508,True,True,True
380,MAHARASHTRA,Mumbai,mumbai,15.096322,17.948685,232.0,1.0,-231.0,231.0,2.852363,2.852363,True,True,True
181,GUJARAT,Jamnagar,jamnagar,13.855961,14.724153,378.0,153.0,-225.0,225.0,0.868192,0.868192,True,True,True
15,ANDHRA PRADESH,Sri Potti Sriramulu Nell*,sri potti sriramulu nell*,15.305836,12.080283,216.0,432.0,216.0,216.0,-3.225553,3.225553,True,True,True
60,ASSAM,Morigaon,morigaon,15.894310,12.597052,167.0,383.0,216.0,216.0,-3.297259,3.297259,True,True,True
520,RAJASTHAN,Udaipur,udaipur,13.340999,14.141332,413.0,207.0,-206.0,206.0,0.800333,0.800333,True,True,True


Rank changes > 10: 475
Rank changes > 25: 271
Rank changes > 50: 158


## Output Manifest

In [11]:
print("Saved outputs:")
for path in [
    PROCESSED_DIR / "dfsi_verified_baseline.csv",
    TABLE_DIR / "dfsi_verified_discrepancy_summary.csv",
    TABLE_DIR / "dfsi_verified_largest_rank_changes.csv",
    TABLE_DIR / "dfsi_verified_top_group_overlap.csv",
    TABLE_DIR / "dfsi_unresolved_records.csv",
]:
    print(f"- {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)")

print("\nNumerical reproduction and ranking reproduction are separate diagnostics:")
print("- numerical reproduction compares DFSI score values directly.")
print("- ranking reproduction compares descending ranks derived from those score values.")

Saved outputs:
- data/processed/dfsi_verified_baseline.csv (208,461 bytes)
- outputs/tables/dfsi_verified_discrepancy_summary.csv (1,374 bytes)
- outputs/tables/dfsi_verified_largest_rank_changes.csv (69,066 bytes)
- outputs/tables/dfsi_verified_top_group_overlap.csv (474 bytes)
- outputs/tables/dfsi_unresolved_records.csv (45,489 bytes)

Numerical reproduction and ranking reproduction are separate diagnostics:
- numerical reproduction compares DFSI score values directly.
- ranking reproduction compares descending ranks derived from those score values.
